BACKGROUND FIT

The goal of this section is to infer background normalization(s) and quantify uncertainties.
We shall also investigate how sensitive is the posterior probability to a difference in priors. 

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.integrate import trapezoid


In [2]:
import sys; sys.path.insert(0, '..')
from src.loader import k_obs, b_nominal, roi_mask

In [ ]:
# roi_mask is imported from loader
K_tot = np.sum(k_obs[roi_mask])
B_tot = np.sum(b_nominal.values[roi_mask])

def log_posterior_bg_only(beta):
    return (K_tot - 1) * np.log(beta) - beta * B_tot

beta_range = np.linspace(0.1, 30, 1000)
log_posterior_values = log_posterior_bg_only(beta_range)

plt.figure()
plt.plot(beta_range, log_posterior_values)
plt.xlabel(r'$\beta$')
plt.ylabel(r'$\log p(\beta|\mathrm{data})$')
plt.title('Log-posterior (scale-invariant prior)')
plt.grid()
plt.tight_layout()
plt.savefig('../plots/log_posterior_scale_invariant.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
posterior_unnorm = np.exp(log_posterior_values) #unnormalized posterior

# Normalize
evidence = trapezoid(posterior_unnorm, beta_range) # integrate over parameter space to get evidence
posterior = posterior_unnorm / evidence


plt.figure()
plt.plot(beta_range, posterior, label='Posterior')
plt.xlabel(r'$\beta$')
plt.ylabel(r'$p(\beta|data)$')
plt.title('Posterior distribution (scale-invariant prior)')
plt.grid()

# Mean
mean_beta = trapezoid(beta_range * posterior, beta_range)

# CDF
cdf = np.cumsum(posterior) * (beta_range[1] - beta_range[0])

# 68% credible interval--> 0.68 = 0.84-0.16
low = beta_range[np.searchsorted(cdf, 0.16)] #Finds the position with 0.16 accumulated probability density
high = beta_range[np.searchsorted(cdf, 0.84)]#same but for 0.84

plt.axvline(mean_beta, color='orange', linestyle='--', label=f'Mean = {mean_beta:.2f}')
plt.fill_between(beta_range, posterior,
                 where=(beta_range >= low) & (beta_range <= high),
                 alpha=0.3, label='68% CI')
plt.legend()
plt.tight_layout()
plt.savefig('../plots/posterior_scale_invariant.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Mean β: {mean_beta:.4f}")
print(f"68% credible interval: [{low:.4f}, {high:.4f}]")

Now let's see what happens when instead of a scale invariant prior we use a flat one. 

In [5]:
beta_min, beta_max = 0.1, 30.0 #This ranges produces a graph that is easier to visuallize. It was found through trial and error. 
prior_flat = 1.0 / (beta_max-beta_min)

In [6]:
#Define log. posterior with the flat prior
def log_posterior_bg_only2(beta):
    return K_tot*np.log(beta)-beta*B_tot+ np.log(prior_flat) #Using the formula from the posterior in the report

In [ ]:
#Now let's sample the posterior for values of beta within the range of 0 and 30 and plot
beta_range = np.linspace(0.1,30,1000)
log_posterior_values = log_posterior_bg_only2(beta_range)

plt.figure()
plt.plot(beta_range, log_posterior_values)
plt.xlabel(r'$\beta$')
plt.ylabel(r'$\log p(\beta|data)$')
plt.title('Log-posterior (flat prior)')
plt.grid()
plt.tight_layout()
plt.savefig('../plots/log_posterior_flat.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
posterior_unnorm = np.exp(log_posterior_values) #unnormalized posterior

# Normalize
evidence = trapezoid(posterior_unnorm, beta_range) # integrate over parameter space to get evidence
posterior = posterior_unnorm / evidence


plt.figure()
plt.plot(beta_range, posterior, label='Posterior')
plt.xlabel(r'$\beta$')
plt.ylabel(r'$p(\beta|data)$')
plt.title('Posterior distribution (flat prior)')
plt.grid()

# Mean
mean_beta = trapezoid(beta_range * posterior, beta_range)

# CDF
cdf = np.cumsum(posterior) * (beta_range[1] - beta_range[0])

# 68% credible interval--> 0.68 = 0.84-0.16
low = beta_range[np.searchsorted(cdf, 0.16)] #Finds the position with 0.16 accumulated probability density
high = beta_range[np.searchsorted(cdf, 0.84)]#same but for 0.84

plt.axvline(mean_beta, color='orange', linestyle='--', label=f'Mean = {mean_beta:.2f}')
plt.fill_between(beta_range, posterior,
                 where=(beta_range >= low) & (beta_range <= high),
                 alpha=0.3, label='68% CI')
plt.legend()
plt.tight_layout()
plt.savefig('../plots/posterior_flat.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Mean β: {mean_beta:.4f}")
print(f"68% credible interval: [{low:.4f}, {high:.4f}]")

The mean normalization for \beta does not seem to be affected much by the prior. 